<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/MovieLens_GCN_Mixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# COMPLETE CELL (fresh notebook): MovieLens-1M KG Deep (Rel-CCN-Att) + Shallow GBRT ensemble
# CHANGE REQUEST APPLIED:
# - Replace ALL NCE / InfoNCE losses with Triplet Hinge Loss (margin=0.2)
# - Keep: lr=1e-3, LAMBDA_HARD=1.0, grad clip=0.5, hard neg curriculum 0->20%, temperature schedule high->low
# - Keep: SentenceTransformer title embeddings, shallow HistGradientBoostingRegressor
# - Report: DEEP and ENSEMBLE Recall@{5,10,20,50}

import os, zipfile, urllib.request, random, re, math
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Repro & device
# ----------------------------
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ----------------------------
# Download / load MovieLens-1M
# ----------------------------
DATA_DIR = "./data"
ML_NAME  = "ml-1m"
ML_DIR   = os.path.join(DATA_DIR, ML_NAME)
ZIP_PATH = os.path.join(DATA_DIR, f"{ML_NAME}.zip")
URL      = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ML_DIR):
    if not os.path.exists(ZIP_PATH):
        print(f"Downloading: {URL}")
        urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

ratings_path = os.path.join(ML_DIR, "ratings.dat")
movies_path  = os.path.join(ML_DIR, "movies.dat")
users_path   = os.path.join(ML_DIR, "users.dat")

ratings_raw = pd.read_csv(
    ratings_path, sep="::", engine="python",
    names=["userId","movieId","rating","timestamp"]
).sort_values(["userId","timestamp"])

movies_df = pd.read_csv(
    movies_path, sep="::", engine="python", encoding="latin-1",
    names=["movieId","title","genres"]
)
users_df = pd.read_csv(
    users_path, sep="::", engine="python",
    names=["userId","gender","age","occupation","zip"]
)

# id maps from FULL ratings
user_ids  = sorted(ratings_raw["userId"].unique().tolist())
movie_ids = sorted(ratings_raw["movieId"].unique().tolist())
uid2u = {uid:i for i,uid in enumerate(user_ids)}
mid2m = {mid:i for i,mid in enumerate(movie_ids)}
num_users, num_movies = len(user_ids), len(movie_ids)
print("num_users:", num_users, "num_movies:", num_movies)

users_sub = users_df[users_df["userId"].isin(user_ids)].copy()
users_sub["u"] = users_sub["userId"].map(uid2u)
movies_sub = movies_df[movies_df["movieId"].isin(movie_ids)].copy()
movies_sub["m"] = movies_sub["movieId"].map(mid2m)

# ----------------------------
# Split: implicit positives + leave-one-out
# ----------------------------
MIN_POS_RATING = 4.0
pos = ratings_raw[ratings_raw["rating"] >= MIN_POS_RATING].copy()
pos["u"] = pos["userId"].map(uid2u)
pos["m"] = pos["movieId"].map(mid2m)
pos.sort_values(["u","timestamp"], inplace=True)

user_hist = defaultdict(list)
for u, m, ts in pos[["u","m","timestamp"]].itertuples(index=False):
    user_hist[u].append((ts, m))
for u in user_hist:
    user_hist[u].sort(key=lambda x: x[0])

train_pos = defaultdict(list)
val_pos, test_pos = {}, {}

for u, seq in user_hist.items():
    ms = [m for _,m in seq]
    if len(ms) >= 3:
        test_pos[u] = ms[-1]
        val_pos[u]  = ms[-2]
        train_pos[u] = ms[:-2]
    elif len(ms) == 2:
        test_pos[u] = ms[-1]
        val_pos[u]  = ms[-2]
        train_pos[u] = []
    elif len(ms) == 1:
        test_pos[u] = ms[-1]
        train_pos[u] = []

train_pos_set = {u: set(train_pos.get(u, [])) for u in range(num_users)}
train_users = [u for u in range(num_users) if len(train_pos.get(u, [])) > 0]
eval_users = [u for u in range(num_users) if u in test_pos]
train_edges = [(u, m) for u in range(num_users) for m in train_pos_set[u]]
print("train_edges:", len(train_edges), "train_users:", len(train_users), "eval_users:", len(eval_users))
if len(train_edges) == 0:
    raise RuntimeError("No training edges. Try lowering MIN_POS_RATING.")

# ----------------------------
# Genres for KG + hard pool
# ----------------------------
genre_set = set()
movie_genre_ids = defaultdict(list)
for m, gstr in movies_sub[["m","genres"]].itertuples(index=False):
    if isinstance(gstr, str) and gstr.strip():
        for g in gstr.split("|"):
            genre_set.add(g)
genres = sorted(list(genre_set))
gid2g = {g:i for i,g in enumerate(genres)}
num_genres = len(genres)

for m, gstr in movies_sub[["m","genres"]].itertuples(index=False):
    if isinstance(gstr, str) and gstr.strip():
        movie_genre_ids[m] = [gid2g[g] for g in gstr.split("|")]
    else:
        movie_genre_ids[m] = []

genre2movies = defaultdict(list)
for m in range(num_movies):
    for gi in movie_genre_ids[m]:
        genre2movies[gi].append(m)

print("num_genres:", num_genres)

# ----------------------------
# User features: one-hot demos + L2 user genre hist + z user train count
# ----------------------------
def zscore(x: np.ndarray):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-6)

gender_vocab = ["F", "M"]  # stable
g2i = {g:i for i,g in enumerate(gender_vocab)}
age_vocab = sorted(users_sub["age"].unique().tolist())
a2i = {int(a):i for i,a in enumerate(age_vocab)}
occ_vocab = sorted(users_sub["occupation"].unique().tolist())
o2i = {int(o):i for i,o in enumerate(occ_vocab)}

G, A, O = len(gender_vocab), len(age_vocab), len(occ_vocab)
user_gender = np.zeros((num_users, G), dtype=np.float32)
user_age    = np.zeros((num_users, A), dtype=np.float32)
user_occ    = np.zeros((num_users, O), dtype=np.float32)

for u, gender, age, occ in users_sub[["u","gender","age","occupation"]].itertuples(index=False):
    gender = str(gender)
    if gender in g2i:
        user_gender[u, g2i[gender]] = 1.0
    user_age[u, a2i[int(age)]] = 1.0
    user_occ[u, o2i[int(occ)]] = 1.0

user_genreh = np.zeros((num_users, num_genres), dtype=np.float32)
user_train_cnt = np.zeros((num_users,), dtype=np.float32)
for u in range(num_users):
    cnt = 0
    for m in train_pos_set[u]:
        for gi in movie_genre_ids[m]:
            user_genreh[u, gi] += 1.0
        cnt += 1
    user_train_cnt[u] = float(cnt)

user_genreh /= (np.linalg.norm(user_genreh, axis=1, keepdims=True) + 1e-12)
cnt_log = np.log1p(user_train_cnt).astype(np.float32)
cnt_min, cnt_max = float(cnt_log.min()), float(cnt_log.max())
user_cnt_z = ((cnt_log - cnt_min) / (cnt_max - cnt_min + 1e-12)).reshape(-1, 1).astype(np.float32)


user_feat = np.concatenate([user_gender, user_age, user_occ, user_genreh, user_cnt_z], axis=1).astype(np.float32)
user_feat_dim = user_feat.shape[1]
user_feat_t = torch.tensor(user_feat, device=device)

# ----------------------------
# Movie features: genres(L2) + pop(min-max) + title embedding (LM)
# ----------------------------
movie_genre_mh = np.zeros((num_movies, num_genres), dtype=np.float32)
for m in range(num_movies):
    for gi in movie_genre_ids[m]:
        movie_genre_mh[m, gi] = 1.0
movie_genre_mh /= (np.linalg.norm(movie_genre_mh, axis=1, keepdims=True) + 1e-12)

movie_train_cnt = np.zeros((num_movies,), dtype=np.float32)
for u in range(num_users):
    for m in train_pos_set[u]:
        movie_train_cnt[m] += 1.0
movie_pop = np.log1p(movie_train_cnt).astype(np.float32)

def minmax(x: np.ndarray):
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-12:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)

pop_mm = minmax(movie_pop).reshape(-1, 1).astype(np.float32)

# Title embedding via SentenceTransformer
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip -q install sentence-transformers
    from sentence_transformers import SentenceTransformer

year_pat = re.compile(r"\(\d{4}\)")
titles = [""] * num_movies
for m, t in movies_sub[["m", "title"]].itertuples(index=False):
    s = str(t)
    s = year_pat.sub("", s).strip()
    s = re.sub(r"\s+", " ", s)
    titles[m] = s

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # 384-d
st = SentenceTransformer(MODEL_NAME, device=str(device))
title_emb = st.encode(
    titles,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

movie_feat = np.concatenate([movie_genre_mh, pop_mm, title_emb], axis=1).astype(np.float32)
movie_feat_dim = movie_feat.shape[1]
movie_feat_t = torch.tensor(movie_feat, device=device)

print("user_feat_dim:", user_feat_dim, "movie_feat_dim:", movie_feat_dim, "title_dim:", title_emb.shape[1])

# ----------------------------
# Node layout + graph (users/movies/genres)
# ----------------------------
OFF_U = 0
OFF_M = OFF_U + num_users
OFF_G = OFF_M + num_movies
num_nodes = OFF_G + num_genres
genre_feat_t = torch.zeros((num_genres, 1), device=device)  # placeholder

REL_U2M = 0
REL_M2U = 1
REL_M2G = 2
REL_G2M = 3
NUM_RELS = 4

edges_src, edges_dst, edges_type = [], [], []
def add_edge(a, b, t):
    edges_src.append(a); edges_dst.append(b); edges_type.append(t)

for u, m in train_edges:
    add_edge(OFF_U+u, OFF_M+m, REL_U2M)
    add_edge(OFF_M+m, OFF_U+u, REL_M2U)

for m in range(num_movies):
    for gi in movie_genre_ids[m]:
        add_edge(OFF_M+m, OFF_G+gi, REL_M2G)
        add_edge(OFF_G+gi, OFF_M+m, REL_G2M)

src = torch.tensor(edges_src, dtype=torch.long, device=device)
dst = torch.tensor(edges_dst, dtype=torch.long, device=device)
etype = torch.tensor(edges_type, dtype=torch.long, device=device)
print("num_nodes:", num_nodes, "num_edges(directed):", src.numel())

# ----------------------------
# Segment softmax
# ----------------------------
def segment_softmax(e, dst, num_nodes):
    max_per_dst = torch.full((num_nodes,), -float("inf"), device=e.device)
    max_per_dst.scatter_reduce_(0, dst, e, reduce="amax", include_self=True)
    e_exp = torch.exp(e - max_per_dst[dst])
    denom = torch.zeros((num_nodes,), device=e.device)
    denom.index_add_(0, dst, e_exp)
    return e_exp / (denom[dst] + 1e-12)

# ----------------------------
# Rel-CCN-Att layer
# ----------------------------
class RelCCNAttLayer(nn.Module):
    def __init__(self, dim, num_rels, dropout=0.15, negative_slope=0.2, add_self_linear=True):
        super().__init__()
        self.dim = dim
        self.num_rels = num_rels
        self.dropout = dropout
        self.negative_slope = negative_slope
        self.W = nn.ModuleList([nn.Linear(dim, dim, bias=False) for _ in range(num_rels)])
        self.r_emb = nn.Embedding(num_rels, dim)
        self.a_l = nn.Parameter(torch.empty(dim))
        self.a_r = nn.Parameter(torch.empty(dim))
        self.a_rel = nn.Parameter(torch.empty(dim))
        self.self_lin = nn.Linear(dim, dim, bias=True) if add_self_linear else None
        self.reset_parameters()

    def reset_parameters(self):
        for w in self.W:
            nn.init.xavier_uniform_(w.weight)
        nn.init.normal_(self.r_emb.weight, std=0.02)
        nn.init.xavier_uniform_(self.a_l.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_r.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_rel.unsqueeze(0))
        if self.self_lin is not None:
            nn.init.xavier_uniform_(self.self_lin.weight)
            nn.init.zeros_(self.self_lin.bias)

    def forward(self, x, src, dst, etype, num_nodes):
        h_r_all = [w(x) for w in self.W]    # list of [N,D]
        rvec = self.r_emb(etype)            # [E,D]

        h_src = torch.empty((src.size(0), self.dim), device=x.device, dtype=x.dtype)
        h_dst = torch.empty((src.size(0), self.dim), device=x.device, dtype=x.dtype)
        for r in range(self.num_rels):
            mask = (etype == r)
            if mask.any():
                hr = h_r_all[r]
                h_src[mask] = hr[src[mask]]
                h_dst[mask] = hr[dst[mask]]

        e = F.leaky_relu(
            (h_src * self.a_l).sum(-1) + (h_dst * self.a_r).sum(-1) + (rvec * self.a_rel).sum(-1),
            negative_slope=self.negative_slope
        )
        alpha = segment_softmax(e, dst, num_nodes)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        msg = h_src + rvec
        out = torch.zeros((num_nodes, self.dim), device=x.device, dtype=x.dtype)
        out.index_add_(0, dst, msg * alpha.unsqueeze(-1))
        if self.self_lin is not None:
            out = out + self.self_lin(x)
        return out

# ----------------------------
# Model: type-specific encoders
# ----------------------------
class KGRecRelAtt(nn.Module):
    def __init__(self, num_nodes, num_users, num_movies, num_genres, off_u, off_m, off_g,
                 user_feat_dim, movie_feat_dim, dim=96, layers=4, dropout=0.15):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, dim)
        nn.init.normal_(self.emb.weight, std=0.01)

        self.user_proj  = nn.Linear(user_feat_dim,  dim, bias=True)
        self.movie_proj = nn.Linear(movie_feat_dim, dim, bias=True)
        self.genre_proj = nn.Linear(1, dim, bias=True)

        for lin in [self.user_proj, self.movie_proj, self.genre_proj]:
            nn.init.xavier_uniform_(lin.weight)
            nn.init.zeros_(lin.bias)

        self.ln0 = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([
            RelCCNAttLayer(dim=dim, num_rels=NUM_RELS, dropout=dropout, add_self_linear=True)
            for _ in range(layers)
        ])

        self.num_users = num_users
        self.num_movies = num_movies
        self.num_genres = num_genres
        self.off_u = off_u
        self.off_m = off_m
        self.off_g = off_g

    def forward(self, src, dst, etype, num_nodes, user_feat_t, movie_feat_t, genre_feat_t):
        x0 = self.emb.weight
        x_user  = self.user_proj(user_feat_t)
        x_movie = self.movie_proj(movie_feat_t)
        x_genre = self.genre_proj(genre_feat_t)

        x0 = x0.clone()
        x0[self.off_u:self.off_u+self.num_users] = x0[self.off_u:self.off_u+self.num_users] + x_user
        x0[self.off_m:self.off_m+self.num_movies] = x0[self.off_m:self.off_m+self.num_movies] + x_movie
        x0[self.off_g:self.off_g+self.num_genres] = x0[self.off_g:self.off_g+self.num_genres] + x_genre

        x = self.ln0(x0)

        xs = [x]
        for layer in self.layers:
            x = layer(x, src, dst, etype, num_nodes)
            x = F.relu(x)
            xs.append(x)

        x_final = torch.mean(torch.stack(xs, dim=0), dim=0)
        user_e  = x_final[self.off_u:self.off_u + self.num_users]
        movie_e = x_final[self.off_m:self.off_m + self.num_movies]
        return user_e, movie_e

model = KGRecRelAtt(
    num_nodes=num_nodes, num_users=num_users, num_movies=num_movies, num_genres=num_genres,
    off_u=OFF_U, off_m=OFF_M, off_g=OFF_G,
    user_feat_dim=user_feat_dim, movie_feat_dim=movie_feat_dim,
    dim=96, layers=4, dropout=0.15
).to(device)

# ----------------------------
# Optim + hyperparams (requested)
# ----------------------------
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)  # lr=1e-3
LAMBDA_HARD = 1.0
CLIP_NORM = 0.5

# ----------------------------
# Negative sampling (no retry loops)
# ----------------------------
all_movies = np.arange(num_movies, dtype=np.int64)

def sample_random_negs(blocked_set, n):
    if len(blocked_set) == 0:
        cand = all_movies
    else:
        blocked_arr = np.fromiter(blocked_set, dtype=np.int64)
        cand = np.setdiff1d(all_movies, blocked_arr, assume_unique=False)
    if cand.size == 0:
        return np.random.randint(0, num_movies, size=n, dtype=np.int64)
    return np.random.choice(cand, size=n, replace=(cand.size < n)).astype(np.int64)

def sample_same_genre_negs(pos_m, blocked_set, n):
    gs = movie_genre_ids[pos_m]
    if not gs:
        return None
    g = max(gs, key=lambda gi: len(genre2movies[gi]))
    pool = np.asarray(genre2movies[g], dtype=np.int64)
    if pool.size <= 1:
        return None
    if len(blocked_set) > 0:
        blocked_arr = np.fromiter(blocked_set, dtype=np.int64)
        cand = np.setdiff1d(pool, blocked_arr, assume_unique=False)
    else:
        cand = pool
    cand = cand[cand != pos_m]
    if cand.size == 0:
        return None
    return np.random.choice(cand, size=n, replace=(cand.size < n)).astype(np.int64)

# ----------------------------
# Curriculum: hard ratio + temperature schedule
# (you requested: hard neg curriculum 0 -> 20%, temperature high -> low)
# ----------------------------
WARMUP_EPOCHS = 5
RAMP_EPOCHS = 15
MAX_HARD_RATIO = 0.20  # 20%

def hard_ratio(epoch_1based: int) -> float:
    if epoch_1based <= WARMUP_EPOCHS:
        return 0.0
    t = (epoch_1based - WARMUP_EPOCHS) / max(1, RAMP_EPOCHS)
    return float(min(MAX_HARD_RATIO, max(0.0, t * MAX_HARD_RATIO)))

# "high -> low" temperature (bigger tau = softer, smaller tau = sharper)
TAU_START = 0.20
TAU_END   = 0.05

def tau_schedule(epoch_1based: int, total_epochs: int) -> float:
    t = (epoch_1based - 1) / max(1, total_epochs - 1)
    cos = 0.5 * (1.0 + math.cos(math.pi * t))
    return float(TAU_END + (TAU_START - TAU_END) * cos)

# ----------------------------
# Triplet hinge losses (UNWEIGHTED), margin=0.2
# Replace ALL InfoNCE/NCE with hinge
# ----------------------------
EPS = 1e-12
HARD_NEG = 10
MARGIN = 0.5

def _cos_sim(a, b, eps=EPS):
    a = F.normalize(a, p=2, dim=-1, eps=eps)
    b = F.normalize(b, p=2, dim=-1, eps=eps)
    return (a * b).sum(dim=-1)

# Triplet hinge losses (NO temperature; NO max-hard; use soft-hard logsumexp)
# - keeps margin semantics stable
# - robust to noisy/false negatives compared to max()
# - beta controls "hardness": bigger -> closer to max, smaller -> closer to mean

def inbatch_triplet_hinge_loss(u_vec, pos_vec, tau: float, margin: float = MARGIN, eps=EPS, beta: float = 10.0):
    """
    In-batch negatives:
      - negatives for i are pos_vec[j], j!=i
      - use SOFT-HARD aggregation via logsumexp(beta * s_neg)/beta instead of max
    Note: tau is intentionally ignored (kept only to avoid changing call sites).
    """
    u = F.normalize(u_vec,  p=2, dim=-1, eps=eps)          # [B,D]
    p = F.normalize(pos_vec, p=2, dim=-1, eps=eps)         # [B,D]
    sim = (u @ p.t())                                      # [B,B] cosine

    s_pos = sim.diag()                                     # [B]
    B = sim.size(0)
    sim_neg = sim.masked_fill(torch.eye(B, device=sim.device, dtype=torch.bool), -1e9)

    # soft-hard negative score per row
    s_neg_soft = torch.logsumexp(beta * sim_neg, dim=1) / beta   # [B]

    loss = F.relu(margin + s_neg_soft - s_pos)
    return loss.mean()

def hardneg_triplet_hinge_loss(u_vec, pos_vec, neg_vec, tau: float, margin: float = MARGIN, eps=EPS, beta: float = 10.0):
    """
    Explicit sampled negatives neg_vec: [B, K, D]
    Use SOFT-HARD aggregation via logsumexp(beta * s_neg)/beta instead of max.
    Note: tau is intentionally ignored (kept only to avoid changing call sites).
    """
    u   = F.normalize(u_vec,   p=2, dim=-1, eps=eps)        # [B,D]
    pos = F.normalize(pos_vec, p=2, dim=-1, eps=eps)        # [B,D]
    neg = F.normalize(neg_vec, p=2, dim=-1, eps=eps)        # [B,K,D]

    s_pos = (u * pos).sum(dim=-1)                           # [B]
    s_neg = (u.unsqueeze(1) * neg).sum(dim=-1)              # [B,K]

    s_neg_soft = torch.logsumexp(beta * s_neg, dim=1) / beta # [B]

    loss = F.relu(margin + s_neg_soft - s_pos)
    return loss.mean()

def triplet_hinge_loss_single_neg(u_vec, pos_vec, neg_vec, tau: float, margin: float = MARGIN, eps=EPS):
    """
    Convenience: when you already have ONE negative per anchor: neg_vec [B,D]
    No temperature; plain hinge.
    Note: tau is intentionally ignored.
    """
    u   = F.normalize(u_vec,   p=2, dim=-1, eps=eps)        # [B,D]
    pos = F.normalize(pos_vec, p=2, dim=-1, eps=eps)        # [B,D]
    neg = F.normalize(neg_vec, p=2, dim=-1, eps=eps)        # [B,D]

    s_pos = (u * pos).sum(dim=-1)                           # [B]
    s_neg = (u * neg).sum(dim=-1)                           # [B]

    loss = F.relu(margin + s_neg - s_pos)
    return loss.mean()

# ----------------------------
# Shallow model: Gradient Boosted Regression (sklearn)
# ----------------------------
from sklearn.ensemble import HistGradientBoostingRegressor

GBRT_NEG_PER_POS = 2
GBRT_MAX_POS = 250_000  # cap for speed

def build_gbrt_dataset():
    Xs = []
    ys = []
    pos_pairs = train_edges
    if len(pos_pairs) > GBRT_MAX_POS:
        pos_pairs = random.sample(pos_pairs, GBRT_MAX_POS)

    for u, m in pos_pairs:
        Xs.append(np.concatenate([user_feat[u], movie_feat[m]], axis=0))
        ys.append(1.0)

    for u, mpos in pos_pairs:
        blocked = set(train_pos_set[u])
        if u in val_pos:  blocked.add(val_pos[u])
        if u in test_pos: blocked.add(test_pos[u])
        blocked.add(mpos)
        negs = sample_random_negs(blocked, GBRT_NEG_PER_POS)
        for mn in negs:
            Xs.append(np.concatenate([user_feat[u], movie_feat[mn]], axis=0))
            ys.append(0.0)

    X = np.stack(Xs, axis=0).astype(np.float32)
    y = np.asarray(ys, dtype=np.float32)
    return X, y

print("\nTraining shallow GBRT...")
X_g, y_g = build_gbrt_dataset()
print("GBRT dataset:", X_g.shape, "pos_rate:", float(y_g.mean()))

gbrt = HistGradientBoostingRegressor(
    loss="squared_error",
    max_depth=6,
    learning_rate=0.1,
    max_iter=200,
    l2_regularization=1e-3,
    random_state=seed
)
gbrt.fit(X_g, y_g)
del X_g, y_g

movie_feat_np = movie_feat.astype(np.float32)
def shallow_scores_for_user(u: int):
    ufeat = user_feat[u].astype(np.float32)
    U = np.repeat(ufeat[None, :], num_movies, axis=0)
    X = np.concatenate([U, movie_feat_np], axis=1)
    return gbrt.predict(X).astype(np.float32)

# ----------------------------
# Recall@K with ensemble
# ----------------------------
@torch.no_grad()
def recall_at_ks_ensemble(user_e, movie_e, ks=(5,10,20,50), users=None, alpha=0.7, eps=EPS):
    if users is None:
        users = eval_users

    user_e  = F.normalize(user_e,  p=2, dim=-1, eps=eps)
    movie_e = F.normalize(movie_e, p=2, dim=-1, eps=eps)
    movie_e_t = movie_e.t()

    recalls = {k: [] for k in ks}
    for u in users:
        gt = test_pos.get(u, None)
        if gt is None:
            continue

        deep_scores = (user_e[u:u+1] @ movie_e_t).squeeze(0).detach().cpu().numpy().astype(np.float32)  # [M]
        shallow_scores = shallow_scores_for_user(u)

        shallow_min, shallow_max = float(shallow_scores.min()), float(shallow_scores.max())
        shallow_norm = (shallow_scores - shallow_min) / (shallow_max - shallow_min + 1e-12)

        deep_norm = (deep_scores + 1.0) * 0.5  # cosine [-1,1] -> [0,1]
        scores = alpha * deep_norm + (1.0 - alpha) * shallow_norm

        seen = set(train_pos_set[u])
        if u in val_pos: seen.add(val_pos[u])
        if len(seen) > 0:
            scores[np.array(list(seen), dtype=np.int64)] = -1e9

        Kmax = min(max(ks), num_movies)
        topk = np.argpartition(-scores, kth=Kmax-1)[:Kmax]
        topk = topk[np.argsort(-scores[topk])]

        for k in ks:
            recalls[k].append(1.0 if gt in topk[:k] else 0.0)

    return {k: float(np.mean(recalls[k])) if len(recalls[k]) else 0.0 for k in ks}

# ----------------------------
# Training (deep) — hinge instead of NCE
# ----------------------------
BATCH = 1024
EPOCHS = 100
steps = max(1, len(train_edges) // BATCH)

print("\nTraining deep model (Triplet Hinge)...")
for epoch in range(1, EPOCHS + 1):
    p_hard = hard_ratio(epoch)
    tau = tau_schedule(epoch, EPOCHS)

    model.train()
    user_e, movie_e = model(src, dst, etype, num_nodes, user_feat_t, movie_feat_t, genre_feat_t)  # one graph forward

    total_loss = 0.0
    hard_used_total = 0

    for _ in range(steps):
        us = np.random.choice(train_users, size=BATCH, replace=True)

        pos_ms = np.empty((BATCH,), dtype=np.int64)
        negs_mx = np.empty((BATCH, HARD_NEG), dtype=np.int64)

        for i, u in enumerate(us):
            pm = int(train_pos[u][np.random.randint(0, len(train_pos[u]))])
            pos_ms[i] = pm

            blocked = set(train_pos_set[u])
            if u in val_pos:  blocked.add(val_pos[u])
            if u in test_pos: blocked.add(test_pos[u])
            blocked.add(pm)

            use_hard = (np.random.rand() < p_hard)
            if use_hard:
                hn = sample_same_genre_negs(pm, blocked, HARD_NEG)
                if hn is None:
                    hn = sample_random_negs(blocked, HARD_NEG)
                hard_used_total += 1
            else:
                hn = sample_random_negs(blocked, HARD_NEG)

            negs_mx[i] = hn

        us_t  = torch.tensor(us, device=device, dtype=torch.long)
        pos_t = torch.tensor(pos_ms, device=device, dtype=torch.long)
        neg_t = torch.tensor(negs_mx, device=device, dtype=torch.long)

        uvec = user_e[us_t]     # [B,D]
        pvec = movie_e[pos_t]   # [B,D]
        nvec = movie_e[neg_t]   # [B,K,D]

        # MAIN: in-batch triplet hinge (hardest in-batch neg)
        loss_main = inbatch_triplet_hinge_loss(uvec, pvec, tau=tau, margin=MARGIN)

        # AUX: sampled neg triplet hinge (hardest among K)
        loss_aux  = hardneg_triplet_hinge_loss(uvec, pvec, nvec, tau=tau, margin=MARGIN)

        loss = loss_main + LAMBDA_HARD * loss_aux
        total_loss += loss

    total_loss = total_loss / steps
    opt.zero_grad(set_to_none=True)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_NORM)
    opt.step()

    # Evaluate deep-only and ensemble
    model.eval()
    ue, me = model(src, dst, etype, num_nodes, user_feat_t, movie_feat_t, genre_feat_t)

    rec_deep = recall_at_ks_ensemble(ue, me, ks=(5,10,20,50), users=eval_users, alpha=1.0)
    rec_ens  = recall_at_ks_ensemble(ue, me, ks=(5,10,20,50), users=eval_users, alpha=0.7)

    used_frac = hard_used_total / (steps * BATCH)
    print(
        f"Epoch {epoch:03d} | tau={tau:.4f} | p_hard={p_hard:.3f} (used {used_frac:.3f}) | "
        f"loss={float(total_loss.detach().cpu()):.4f} | "
        f"DEEP R@5={rec_deep[5]:.4f} R@10={rec_deep[10]:.4f} R@20={rec_deep[20]:.4f} R@50={rec_deep[50]:.4f} | "
        f"ENS(alpha=0.7) R@5={rec_ens[5]:.4f} R@10={rec_ens[10]:.4f} R@20={rec_ens[20]:.4f} R@50={rec_ens[50]:.4f}"
    )

print("\nDone.")

device: cuda
num_users: 6040 num_movies: 3706
train_edges: 563206 train_users: 6035 eval_users: 6038
num_genres: 18


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

user_feat_dim: 49 movie_feat_dim: 403 title_dim: 384
num_nodes: 9764 num_edges(directed): 1138796

Training shallow GBRT...
GBRT dataset: (750000, 452) pos_rate: 0.3333333432674408

Training deep model (Triplet Hinge)...
Epoch 001 | tau=0.2000 | p_hard=0.000 (used 0.000) | loss=1.9533 | DEEP R@5=0.0038 R@10=0.0071 R@20=0.0118 R@50=0.0272 | ENS(alpha=0.7) R@5=0.0346 R@10=0.0542 R@20=0.0922 R@50=0.1838
Epoch 002 | tau=0.2000 | p_hard=0.000 (used 0.000) | loss=1.9061 | DEEP R@5=0.0113 R@10=0.0190 R@20=0.0354 R@50=0.0744 | ENS(alpha=0.7) R@5=0.0331 R@10=0.0528 R@20=0.0918 R@50=0.1855
Epoch 003 | tau=0.1998 | p_hard=0.000 (used 0.000) | loss=1.8776 | DEEP R@5=0.0144 R@10=0.0283 R@20=0.0494 R@50=0.0992 | ENS(alpha=0.7) R@5=0.0316 R@10=0.0530 R@20=0.0896 R@50=0.1820
Epoch 004 | tau=0.1997 | p_hard=0.000 (used 0.000) | loss=1.8572 | DEEP R@5=0.0204 R@10=0.0333 R@20=0.0570 R@50=0.1164 | ENS(alpha=0.7) R@5=0.0316 R@10=0.0527 R@20=0.0899 R@50=0.1805
Epoch 005 | tau=0.1994 | p_hard=0.000 (used 0.0